In [1]:
import pandas as pd

In [ ]:
# ## ----- the directory convention used is based off the google drive organization ------ ##

# check og df

In [2]:
# rheo_df = pd.read_excel("/Users/mauriellenoto/Desktop/seager/ionicliquids/isn/UAFW/UAFW_20260707_rheo.xls")

import pandas as pd

tga_xl = pd.ExcelFile('/Users/mauriellenoto/Desktop/seager/ionicliquids/isn/TGA/2026-07-08/2026-07-08-TGA-UAFW.xls',
                    #    sheet_name="Ramp 10.00 °Cmin to 650.00 °C", 
                     #   header=1
                       )
# cols = list(dsc_df.columns)
# print(cols)
tga_xl

# print(type(cols))
# sheets = list(tga_df.keys())
sheets = tga_xl.sheet_names
sheets
# data = sheets[1:]
# sheets


['Details', 'Ramp 10.00 °Cmin to 800.00 °C']

# extract data fxn

In [2]:

# if you want a df of the details, use:
# details_df = pd.read_excel("path_to_excel_file", sheet_name = "Details", header = 1)

def extract_data(path): 
    # Read in original file 
    xl = pd.ExcelFile(path) 
    all_sheets = xl.sheet_names 
    print(all_sheets)
    
    data_sheets = all_sheets[1:] 
    df_list = [] 
    
    for sheet in data_sheets: 
        sheet_df = pd.read_excel(path, sheet_name=sheet, header=1) 
        
        # extract the units from the first row of data (index 0)
        units = sheet_df.iloc[0].fillna('').astype(str).tolist()
        
        # combine old column names with the units
        new_columns = []
        for col, unit in zip(sheet_df.columns, units):
            clean_col = col.split('.')[0] if '.' in col else col
            if unit:
                new_columns.append(f"{clean_col} ({unit})")
            else:
                new_columns.append(clean_col)
                
        # assign the new combined names back to the dataframe columns
        sheet_df.columns = new_columns
        
        # drop units row from the data
        sheet_df = sheet_df.drop(index=0).reset_index(drop=True)
        
        # add source sheet column
        sheet_df['Source Sheet'] = sheet 
        df_list.append(sheet_df) 
        
    # Vertically stack all the sheets together into one final df 
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # convert to float
    # errors='coerce' turns any unconvertible text or bad data into NaN safely
    for col in combined_df.columns:
        if col != 'Source Sheet':
            combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')
    
    # make sure source sheet is last column in final df
    cols = [col for col in combined_df.columns if col != 'Source Sheet'] + ['Source Sheet']
    combined_df = combined_df[cols]
    
    return combined_df

# plotting

In [ ]:
import matplotlib.pyplot as plt

def plot_multiple(df_list, x_col, y1_col, y2_col, titles=None, global_title=None):
    """
    Plots multiple DSC datasets in a grid of subplots using twin y-axes.
    
    Parameters:
    - df_list: List of pandas DataFrames (e.g., [df1, df2, df3])
    - x_col: Name of the column for the shared X axis (Temperature)
    - y1_col: Name of the column for the left Y axis (Heat Flow W/g)
    - y2_col: Name of the column for the right Y axis (Heat Flow J/gC)
    - titles: Optional list of strings for each subplot title
    """
    num_plots = len(df_list)
    if num_plots == 0:
        return
    

    nrows = 1 if num_plots <= 2 else 2
    ncols = num_plots if num_plots <= 2 else 2
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(10,4))
    

    if num_plots == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    

    for i, df in enumerate(df_list):
        ax1 = axes[i]
        
        ax1.plot(df[x_col], df[y1_col], color="magenta", lw=2, label=y1_col)
        ax1.set_xlabel(x_col)
        ax1.set_ylabel(y1_col, color="magenta")
        ax1.tick_params(axis='y', labelcolor="magenta")
        
        ax2 = ax1.twinx()
        ax2.plot(df[x_col], df[y2_col], color="orange", lw=2, label=y2_col)
        ax2.set_ylabel(y2_col, color="orange")
        ax2.tick_params(axis='y', labelcolor="orange")
        
        # set individual subplot title
        if titles and i < len(titles):
            ax1.set_title(titles[i], fontsize=12)
        else:
            ax1.set_title(f"Dataset {i+1}", fontsize=12)
            
    # take out fourth subplot if three are passed in
    for j in range(num_plots, len(axes)):
        fig.delaxes(axes[j])

    # add suptitle
    if global_title:
        fig.suptitle(global_title, fontsize=16, fontweight='bold')
        # rect=[0, 0, 1, 0.95] keeps the subplots from crashing into the suptitle
        plt.tight_layout(rect=[0, 0, 1, 0.95])
    else:
        plt.tight_layout()
        
    plt.tight_layout()
    plt.show()


In [ ]:
# dictionary for plotting

dfs_and_details = {
                    "Rhemoetry":
                        # data info
                            {"path1": "/Users/mauriellenoto/Desktop/seager/ionicliquids/data/RHEO/2026-07-26/20260726-RHEO-UAFW.xls",
                             "date": "2026-07-26",
                             "chemical": "UAFW",
                        # y-axis colors
                             "y1_color": "yellowgreen",
                             "y2_color": "mediumorchid",
                        # axis limits
                             "lims":{
                                 "xlims":
                                        # None
                                        {
                                        "xmin": None,
                                        "xmax": None, 
                                        },
                                "y1lims":
                                        # None
                                        {
                                        "y1min": None,
                                        "y1max": None, 
                                        },

                                "y2lims":
                                        # None
                                        {
                                        "y2min": None,
                                        "y2max": None, 
                                        },
                             }
                             
                            }
                    }

In [17]:
dict = {"lims":{
                                 "xlims":
                                        # None
                                        {
                                        "xmin": None,
                                        "xmax": None, 
                                        },
                                "y1lims":
                                        # None
                                        {
                                        "y1min": None,
                                        "y1max": None, 
                                        },

                                "y2lims":
                                        # None
                                        {
                                        "y2min": None,
                                        "y2max": None, 
                                        },
                             }}


KeyError: 0